In [ ]:
# -*- coding: utf-8 -*-
# ===========================================================
# 전체 데모 테스트 코드 (MLP + TabTransformer + RealMLP)
# ===========================================================

# Standard Library
from abc import ABC, abstractmethod
import copy

# Third Party
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score
from torch.utils.data import DataLoader, TensorDataset


# -----------------------------------------------------------
# Base Neural Network Model
# -----------------------------------------------------------
class BaseNNModel(nn.Module, ABC):
    @abstractmethod
    def __init__(self, **kwargs):
        super(BaseNNModel, self).__init__()

    @abstractmethod
    def build_network(self) -> nn.Module:
        pass

    @abstractmethod
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        pass


# -----------------------------------------------------------
# MLP Model
# -----------------------------------------------------------
class MLPModel(BaseNNModel):
    def __init__(
        self,
        input_dim: int,
        hidden_dims: list[int],
        cat_features: list[int] | None = None,
        cat_dims: list[int] | None = None,
        emb_dim: int = 8,
    ):
        super(MLPModel, self).__init__()

        self.input_dim = input_dim
        self.hidden_dims = hidden_dims
        self.cat_features = cat_features or []
        self.cat_dims = cat_dims or []
        self.emb_dim = emb_dim

        self.embeddings = None
        self.network = self.build_network()

    def build_network(self) -> nn.Sequential:
        if len(self.cat_dims) > 0:
            self.embeddings = nn.ModuleList(
                [nn.Embedding(cat_dim, self.emb_dim) for cat_dim in self.cat_dims]
            )

        combined_input_dim = (
            self.input_dim - len(self.cat_features)
            + len(self.cat_features) * self.emb_dim
        )

        layers = []
        dims = [combined_input_dim] + self.hidden_dims

        for i in range(len(dims) - 1):
            hidden_layer = nn.Linear(dims[i], dims[i + 1])
            nn.init.kaiming_normal_(hidden_layer.weight, mode="fan_in", nonlinearity="relu")

            layers.append(hidden_layer)
            layers.append(nn.BatchNorm1d(dims[i + 1]))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.2))

        layers.append(nn.Linear(dims[-1], 1))
        return nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        continuous_features = []
        embedded_features = []
        cat_idx = 0

        for i in range(self.input_dim):
            if i in self.cat_features:
                cat_values = x[:, i].long()
                embedded = self.embeddings[cat_idx](cat_values)
                embedded_features.append(embedded)
                cat_idx += 1
            else:
                continuous_features.append(x[:, i : i + 1])

        continuous_features = torch.cat(continuous_features, dim=1) if continuous_features else None
        embedded_features = torch.cat(embedded_features, dim=1) if embedded_features else None

        if continuous_features is not None and embedded_features is not None:
            combined_features = torch.cat([continuous_features, embedded_features], dim=1)
        elif embedded_features is not None:
            combined_features = embedded_features
        elif continuous_features is not None:
            combined_features = continuous_features
        else:
            raise ValueError("No features found for forward pass.")

        logits = self.network(combined_features)
        return logits


# -----------------------------------------------------------
# TabTransformer Model (당신 코드 그대로)
# -----------------------------------------------------------
class TabTransformerModel(BaseNNModel):
    def __init__(
        self,
        input_dim: int,
        cat_features: list[int] | None = None,
        cat_dims: list[int] | None = None,
        embedding_dropout_ratio: float = 0.1,
        padding_idx: int | None = None,
        add_cls: bool = False,
        d_token: int = 32,
        n_heads: int = 4,
        dim_feedforward: int = 128,
        attn_dropout_ratio: float = 0.1,
        n_layers: int = 2,
        pooling: str = None,  # "concat" or "cls" or "mean"
        cont_transform: str = "linear",  # "none" or "linear"
        mlp_hidden_dims: tuple[int, ...] = (128, 64),
        mlp_dropout_ratio: float = 0.2,
    ):
        super().__init__()
        self.input_dim = input_dim
        self.cat_features = cat_features or []
        self.cat_dims = cat_dims or []
        self.embedding_dropout_ratio = embedding_dropout_ratio
        self.padding_idx = padding_idx
        self.add_cls = add_cls
        self.d_token = d_token
        self.n_heads = n_heads
        self.dim_feedforward = dim_feedforward
        self.attn_dropout_ratio = attn_dropout_ratio
        self.n_layers = n_layers
        self.pooling = pooling
        self.cont_transform = cont_transform
        self.mlp_hidden_dims = mlp_hidden_dims
        self.mlp_dropout_ratio = mlp_dropout_ratio

        self.cat_embeddings = None
        self.col_embedding = None
        self.cls_token = None
        self.embedding_dropout = None
        self.transformer = None
        self.mlp = None
        self.cont_bn = None
        self.cont_linear = None

        self.build_network()

    def build_network(self) -> None:
        if len(self.cat_features) != len(self.cat_dims):
            raise ValueError(
                f"Mismatch between 'cat_features' and 'cat_dims'. "
                f"Length of cat_features: {len(self.cat_features)}, "
                f"Length of cat_dims: {len(self.cat_dims)}"
            )

        n_cat = len(self.cat_features)
        n_cont = self.input_dim - n_cat

        self.register_buffer(
            "cat_idx_tensor",
            (torch.tensor(self.cat_features, dtype=torch.long)
             if self.cat_features else torch.empty(0, dtype=torch.long)),
        )

        cont_idx = [i for i in range(self.input_dim) if i not in self.cat_features]
        self.register_buffer(
            "cont_idx_tensor",
            (torch.tensor(cont_idx, dtype=torch.long)
             if cont_idx else torch.empty(0, dtype=torch.long)),
        )

        # categorical path
        if n_cat == 0:
            self.cat_embeddings = nn.ModuleList()
            self.col_embedding = None
        else:
            self.cat_embeddings = nn.ModuleList(
                [
                    nn.Embedding(
                        num_embeddings=c + (1 if self.padding_idx is not None else 0),
                        embedding_dim=self.d_token,
                        padding_idx=(self.padding_idx if self.padding_idx is not None else None),
                    )
                    for c in self.cat_dims
                ]
            )
            self.col_embedding = nn.Embedding(num_embeddings=n_cat, embedding_dim=self.d_token)

        if self.add_cls:
            self.cls_token = nn.Parameter(torch.zeros(1, 1, self.d_token))
            nn.init.normal_(self.cls_token, std=0.02)

        self.embedding_dropout = nn.Dropout(self.embedding_dropout_ratio)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.d_token,
            nhead=self.n_heads,
            dim_feedforward=self.dim_feedforward,
            dropout=self.attn_dropout_ratio,
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=self.n_layers)

        for emb in self.cat_embeddings:
            nn.init.normal_(emb.weight, std=0.02)
        if self.col_embedding is not None:
            nn.init.normal_(self.col_embedding.weight, std=0.02)

        cat_feature_dim = (self.d_token if self.pooling == "cls" else n_cat * self.d_token)

        # continuous path
        if n_cont > 0:
            self.cont_bn = nn.BatchNorm1d(n_cont)
            if self.cont_transform == "linear":
                self.cont_linear = nn.Linear(n_cont, self.d_token)
                nn.init.kaiming_uniform_(self.cont_linear.weight, mode="fan_in", nonlinearity="relu")
                cont_feature_dim = self.d_token
            else:
                self.cont_linear = nn.Identity()
                cont_feature_dim = n_cont
        else:
            self.cont_bn = None
            self.cont_linear = None
            cont_feature_dim = 0

        # head
        mlp_input_dim = cat_feature_dim + cont_feature_dim
        layers = []
        input_size = mlp_input_dim
        for hidden_size in self.mlp_hidden_dims:
            linear_layer = nn.Linear(input_size, hidden_size)
            nn.init.kaiming_uniform_(linear_layer.weight, mode="fan_in", nonlinearity="relu")
            layers.extend([linear_layer, nn.BatchNorm1d(hidden_size), nn.ReLU(), nn.Dropout(self.mlp_dropout_ratio)])
            input_size = hidden_size
        layers.append(nn.Linear(input_size, 1))
        self.mlp = nn.Sequential(*layers)

    def _encode_categoricals(self, x_cat: torch.LongTensor) -> torch.Tensor:
        batch_size = x_cat.size(0)
        n_cat = len(self.cat_features)
        if n_cat == 0:
            return torch.zeros(
                batch_size,
                self.d_token if self.pooling == "cls" else 0,
                device=x_cat.device,
                dtype=torch.float32,
            )

        tok_list = []
        for j, emb in enumerate(self.cat_embeddings):
            tok = emb(x_cat[:, j])
            if self.col_embedding is not None:
                tok = tok + self.col_embedding.weight[j]
            tok_list.append(tok.unsqueeze(1))

        x_tok = torch.cat(tok_list, dim=1)

        if self.add_cls:
            cls = self.cls_token.expand(batch_size, -1, -1)
            x_tok = torch.cat([cls, x_tok], dim=1)

        x_tok = self.embedding_dropout(x_tok)
        z = self.transformer(x_tok)

        if self.pooling == "cls" and self.add_cls:
            out = z[:, 0, :]
        elif self.pooling == "mean":
            out = z.mean(dim=1)
        else:
            if self.add_cls:
                z = z[:, 1:, :]
            out = z.reshape(batch_size, -1)
        return out

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size = x.shape[0]
        n_cat = len(self.cat_features)
        n_cont = self.input_dim - n_cat

        if n_cat > 0:
            x_cat = x[:, self.cat_idx_tensor].long()
            if self.padding_idx is not None:
                x_cat = x_cat + 1
        else:
            x_cat = torch.zeros(batch_size, 0, dtype=torch.long, device=x.device)

        z_cat = self._encode_categoricals(x_cat)

        if n_cont > 0:
            x_cont = x[:, self.cont_idx_tensor].float()
            if self.cont_bn is not None:
                x_cont = self.cont_bn(x_cont)
            x_cont = self.cont_linear(x_cont)
            z = torch.cat([z_cat, x_cont], dim=1)
        else:
            z = z_cat

        logits = self.mlp(z)
        return logits


# -----------------------------------------------------------
# RealMLP (TD-S 스타일) - 전처리 유틸
# -----------------------------------------------------------
def smooth_clip_np(x: np.ndarray) -> np.ndarray:
    # f(x) = x / (1 + |x|^3)^(2/3)
    return x / np.power(1.0 + np.abs(x) ** 3, 2.0 / 3.0)

def fit_robust_scale_params(X_cont: np.ndarray, eps: float = 1e-12):
    """
    column-wise robust scaling params.
    median=q50, scale:
      if q75!=q25: 1/(q75-q25)
      else if q50!=q0: 2/(q50-q0)
      else: 0
    """
    q0  = np.quantile(X_cont, 0.00, axis=0)
    q25 = np.quantile(X_cont, 0.25, axis=0)
    q50 = np.quantile(X_cont, 0.50, axis=0)
    q75 = np.quantile(X_cont, 0.75, axis=0)

    iqr = q75 - q25
    scale = np.zeros_like(iqr, dtype=np.float32)

    mask1 = np.abs(iqr) > eps
    scale[mask1] = 1.0 / iqr[mask1]

    mask2 = (~mask1) & (np.abs(q50 - q0) > eps)
    scale[mask2] = 2.0 / (q50[mask2] - q0[mask2])

    median = q50.astype(np.float32)
    scale = scale.astype(np.float32)
    return median, scale


# -----------------------------------------------------------
# RealMLP (TD-S 스타일) - 모델
# -----------------------------------------------------------
class DiagonalScaling(nn.Module):
    def __init__(self, dim: int, init: float = 1.0):
        super().__init__()
        self.scale = nn.Parameter(torch.full((dim,), float(init)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.scale


class RealMLPModel(BaseNNModel):
    def __init__(
        self,
        input_dim: int,
        cat_features: list[int] | None = None,
        cat_dims: list[int] | None = None,
        cont_median: list[float] | None = None,
        cont_scale: list[float] | None = None,
        hidden_width: int = 256,
        n_hidden_layers: int = 3,
        activation: str = "selu",  # "selu" or "mish"
        encode_binary_as_pm1: bool = True,
    ):
        super().__init__()
        self.input_dim = input_dim
        self.cat_features = cat_features or []
        self.cat_dims = cat_dims or []
        self.encode_binary_as_pm1 = encode_binary_as_pm1

        if len(self.cat_features) != len(self.cat_dims):
            raise ValueError("cat_features and cat_dims length mismatch")

        # index buffers
        self.register_buffer(
            "cat_idx_tensor",
            torch.tensor(self.cat_features, dtype=torch.long) if self.cat_features else torch.empty(0, dtype=torch.long),
        )
        cont_idx = [i for i in range(self.input_dim) if i not in self.cat_features]
        self.register_buffer(
            "cont_idx_tensor",
            torch.tensor(cont_idx, dtype=torch.long) if cont_idx else torch.empty(0, dtype=torch.long),
        )
        self.n_cont = len(cont_idx)

        # preprocessing buffers (continuous)
        if self.n_cont > 0:
            if cont_median is None or cont_scale is None:
                cont_median = [0.0] * self.n_cont
                cont_scale = [1.0] * self.n_cont
            if len(cont_median) != self.n_cont or len(cont_scale) != self.n_cont:
                raise ValueError(
                    f"cont_median/cont_scale length must match n_cont={self.n_cont}. "
                    f"Got median={len(cont_median)}, scale={len(cont_scale)}"
                )
            self.register_buffer("cont_median", torch.tensor(cont_median, dtype=torch.float32))
            self.register_buffer("cont_scale", torch.tensor(cont_scale, dtype=torch.float32))
        else:
            self.cont_median = None
            self.cont_scale = None

        # one-hot dim
        onehot_dim = 0
        for c in self.cat_dims:
            if self.encode_binary_as_pm1 and c == 2:
                onehot_dim += 1
            else:
                onehot_dim += int(c)
        self.onehot_dim = onehot_dim
        self.d_in = self.onehot_dim + self.n_cont

        # activation
        act = activation.lower().strip()
        if act == "selu":
            self.act_name = "selu"
            self.act = nn.SELU()
        elif act == "mish":
            self.act_name = "mish"
            self.act = nn.Mish()
        else:
            raise ValueError("activation must be 'selu' or 'mish'")

        self.diag = DiagonalScaling(self.d_in, init=1.0)
        self.mlp = self.build_network(hidden_width, n_hidden_layers)
        self._init_weights()

    def build_network(self, hidden_width: int, n_hidden_layers: int) -> nn.Module:
        layers: list[nn.Module] = []
        in_dim = self.d_in
        for _ in range(n_hidden_layers):
            layers.append(nn.Linear(in_dim, hidden_width))
            layers.append(self.act)
            in_dim = hidden_width
        layers.append(nn.Linear(in_dim, 1))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                if self.act_name == "selu":
                    # LeCun normal-ish
                    nn.init.normal_(m.weight, mean=0.0, std=(1.0 / m.in_features) ** 0.5)
                else:
                    nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")
                nn.init.zeros_(m.bias)

    @staticmethod
    def _smooth_clip_torch(x: torch.Tensor) -> torch.Tensor:
        return x / torch.pow(1.0 + torch.abs(x) ** 3, 2.0 / 3.0)

    def _preprocess_cont(self, x_cont: torch.Tensor) -> torch.Tensor:
        if x_cont.numel() == 0:
            return x_cont
        xc = (x_cont - self.cont_median) * self.cont_scale
        return self._smooth_clip_torch(xc)

    def _encode_categorical_onehot(self, x_cat: torch.Tensor) -> torch.Tensor:
        if x_cat.numel() == 0:
            return torch.zeros((x_cat.shape[0], 0), device=x_cat.device, dtype=torch.float32)

        outs = []
        for j, card in enumerate(self.cat_dims):
            col = x_cat[:, j]
            if self.encode_binary_as_pm1 and card == 2:
                pm1 = col.to(torch.float32) * 2.0 - 1.0
                outs.append(pm1.unsqueeze(1))
            else:
                oh = F.one_hot(col.clamp(min=0, max=card - 1), num_classes=card).to(torch.float32)
                outs.append(oh)
        return torch.cat(outs, dim=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B = x.shape[0]
        n_cat = len(self.cat_features)

        if n_cat > 0:
            x_cat = x[:, self.cat_idx_tensor].long()
        else:
            x_cat = torch.zeros((B, 0), device=x.device, dtype=torch.long)

        if self.n_cont > 0:
            x_cont = x[:, self.cont_idx_tensor].to(torch.float32)
            x_cont = self._preprocess_cont(x_cont)
        else:
            x_cont = torch.zeros((B, 0), device=x.device, dtype=torch.float32)

        x_oh = self._encode_categorical_onehot(x_cat)
        feats = torch.cat([x_oh, x_cont], dim=1)
        feats = self.diag(feats)
        logits = self.mlp(feats)
        return logits


# -----------------------------------------------------------
# Deep Learning Binary Classifier
# -----------------------------------------------------------
class DeepLearningBinaryClassifier(BaseEstimator, ClassifierMixin):
    def __init__(
        self,
        model_type: str = "mlp",
        model_params: dict | None = None,
    ):
        self.model_type = model_type
        self.model_params = model_params or {}
        self.model = None

    @property
    def device(self) -> torch.device:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def _build_model(self) -> BaseNNModel:
        model_registry = {
            "mlp": MLPModel,
            "tabtransformer": TabTransformerModel,
            "realmlp": RealMLPModel,
        }

        if self.model_type not in model_registry:
            raise ValueError(f"Unknown model type: {self.model_type}")

        valid_params = {k: v for k, v in self.model_params.items() if k not in ["loss_fn", "lr", "weight_decay"]}
        model_class = model_registry[self.model_type](**valid_params)
        return model_class

    def _get_loss_fn(self) -> nn.Module:
        loss_name = self.model_params.get("loss_fn", "logloss")
        if loss_name == "logloss":
            return nn.BCEWithLogitsLoss(reduction="none")
        else:
            raise ValueError(f"Unknown loss function: {loss_name}")

    def fit(
        self,
        X: np.ndarray,
        y: np.ndarray,
        sample_weight: np.ndarray | None = None,
        eval_set: list[tuple[np.ndarray, np.ndarray]] | None = None,
        eval_metric: list[str] | None = None,
        max_epochs: int = 10,
        patience: int | None = None,
        batch_size: int = 128,
        verbose: bool = True,
    ) -> "DeepLearningBinaryClassifier":

        lr = self.model_params.get("lr", 0.001)
        weight_decay = self.model_params.get("weight_decay", 1e-4)  # ← 추가 (RealMLP는 0.0 권장)
        eval_metric = eval_metric or ["logloss"]

        X_tensor = torch.tensor(X, dtype=torch.float32).to(self.device)
        y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1).to(self.device)

        if eval_set is not None:
            x_eval_tensor = torch.tensor(eval_set[0][0], dtype=torch.float32).to(self.device)
            y_eval_true = eval_set[0][1]
        else:
            x_eval_tensor = None
            y_eval_true = None

        if sample_weight is not None:
            sample_weight_tensor = torch.tensor(sample_weight, dtype=torch.float32).view(-1, 1).to(self.device)
        else:
            sample_weight_tensor = torch.ones_like(y_tensor, dtype=torch.float32)

        train_dataset = TensorDataset(X_tensor, y_tensor, sample_weight_tensor)
        train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

        if self.model is None:
            self.model = self._build_model().to(self.device)

        loss_fn = self._get_loss_fn()
        optimizer = optim.Adam(self.model.parameters(), lr=lr, weight_decay=weight_decay)

        patience_counter = 0
        best_metric = float("inf")
        best_model_weights = None

        for epoch in range(max_epochs):
            self.model.train()
            epoch_loss = 0.0
            n_batches = 0

            for x_batch, y_batch, weight_batch in train_dataloader:
                optimizer.zero_grad()
                y_pred_logits = self.model(x_batch)
                loss = loss_fn(y_pred_logits, y_batch)
                weighted_loss = (loss * weight_batch).sum() / weight_batch.sum()
                weighted_loss.backward()
                optimizer.step()

                epoch_loss += weighted_loss.item()
                n_batches += 1

            if verbose:
                print(f"Epoch {epoch + 1}/{max_epochs} - [train] loss: {epoch_loss / max(1, n_batches):.6f}")

            if eval_set is not None:
                self.model.eval()
                with torch.no_grad():
                    y_eval_logits = self.model(x_eval_tensor)
                    y_eval_pred = torch.sigmoid(y_eval_logits).cpu().numpy().ravel()

                eval_metrics = {}
                for metric in eval_metric:
                    if metric == "logloss":
                        eval_metrics["logloss"] = log_loss(y_eval_true, y_eval_pred)
                    elif metric == "average_precision":
                        eval_metrics["average_precision"] = -average_precision_score(y_eval_true, y_eval_pred)
                    elif metric == "auc":
                        eval_metrics["auc"] = -roc_auc_score(y_eval_true, y_eval_pred)
                    else:
                        raise ValueError(f"Unknown metric: {metric}")

                if verbose:
                    metrics_str = ", ".join([f"{k}: {v:.4f}" for k, v in eval_metrics.items()])
                    print(f"  - [eval] {metrics_str}")

                main_metric_name = eval_metric[0]
                current_metric = eval_metrics.get(main_metric_name, eval_metrics["logloss"])

                if verbose:
                    print(f"    -- (early_stopping) current_metric: {current_metric:.6f}, best_metric: {best_metric:.6f}")

                if current_metric < best_metric:
                    best_metric = current_metric
                    patience_counter = 0
                    best_model_weights = copy.deepcopy(self.model.state_dict())
                else:
                    patience_counter += 1
                    if patience is not None and patience_counter >= patience:
                        if verbose:
                            print(f"Early stopping at epoch {epoch + 1}")
                        break

        if best_model_weights is not None:
            self.model.load_state_dict(best_model_weights)

        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        self.model = self.model.to(self.device)
        X_tensor = torch.tensor(X, dtype=torch.float32).to(self.device)

        with torch.no_grad():
            self.model.eval()
            logits = self.model(X_tensor)
            probs1 = torch.sigmoid(logits).cpu().numpy()

        if probs1.shape[1] == 1:
            probs1 = probs1.reshape(-1, 1)

        probs0 = 1.0 - probs1
        probs = np.hstack((probs0, probs1))
        return probs.astype("float")

    def predict(self, X: np.ndarray) -> np.ndarray:
        probs = self.predict_proba(X)
        return probs.argmax(axis=1)


# -----------------------------------------------------------
# 더미 데이터셋 (당신 코드 그대로)
# -----------------------------------------------------------
def make_kor_feature_demo_data(n_samples: int = 5000, seed: int = 42):
    rng = np.random.RandomState(seed)
    n = n_samples

    age = np.clip(rng.normal(loc=40, scale=12, size=n), 18, 80).astype("float32")
    gender = rng.randint(0, 3, size=n).astype("int64")
    app_join_cnt = rng.poisson(lam=2.0, size=n).astype("float32")
    last_login_days = rng.exponential(scale=10.0, size=n).astype("float32")
    os_cat = rng.randint(0, 4, size=n).astype("int64")
    app_join_days = rng.exponential(scale=200.0, size=n).astype("float32")

    biz_owner_score = (rng.binomial(1, 0.3, size=n) * rng.uniform(0.5, 1.0, size=n)).astype("float32")
    home_owner_score = (rng.binomial(1, 0.4, size=n) * rng.uniform(0.5, 1.0, size=n)).astype("float32")
    car_owner_score = (rng.binomial(1, 0.5, size=n) * rng.uniform(0.5, 1.0, size=n)).astype("float32")
    married_score   = (rng.binomial(1, 0.5, size=n) * rng.uniform(0.5, 1.0, size=n)).astype("float32")
    child_score     = (rng.binomial(1, 0.4, size=n) * rng.uniform(0.5, 1.0, size=n)).astype("float32")

    w_gender = rng.uniform(-0.3, 0.3, size=3)
    w_os     = rng.uniform(-0.2, 0.2, size=4)

    score = np.zeros(n, dtype="float32")
    score += 0.05 * (age - 40) / 10.0
    score += 0.25 * app_join_cnt
    score -= 0.03 * last_login_days
    score -= 0.002 * app_join_days
    score += 0.8 * biz_owner_score
    score += 0.6 * home_owner_score
    score += 0.7 * car_owner_score
    score += 0.9 * married_score
    score += 1.0 * child_score
    score += w_gender[gender]
    score += w_os[os_cat]

    noise = rng.normal(scale=0.5, size=n).astype("float32")
    bias = -0.2
    logit = score + bias + noise
    prob = 1.0 / (1.0 + np.exp(-logit))
    y = (prob > 0.5).astype("int64")

    X = np.column_stack(
        [
            age,                       # 0
            gender.astype("float32"),  # 1 (cat)
            app_join_cnt,              # 2
            last_login_days,           # 3
            os_cat.astype("float32"),  # 4 (cat)
            app_join_days,             # 5
            biz_owner_score,           # 6
            home_owner_score,          # 7
            car_owner_score,           # 8
            married_score,             # 9
            child_score,               # 10
        ]
    ).astype("float32")

    cat_feature_indices = [1, 4]
    cat_dims = [3, 4]
    return X, y, cat_feature_indices, cat_dims


# -----------------------------------------------------------
# RealMLP 데모 (더미 데이터로 바로 학습/평가)
# -----------------------------------------------------------
def demo_train_realmlp():
    print("\n===== RealMLP Demo (KOR Feature Schema) =====")

    X, y, cat_features, cat_dims = make_kor_feature_demo_data(n_samples=4000, seed=777)
    print("X shape:", X.shape, " | y shape:", y.shape)
    print("Categorical feature indices:", cat_features)
    print("Categorical dims (cardinality):", cat_dims)

    # split
    N = X.shape[0]
    idx = np.arange(N)
    np.random.shuffle(idx)

    tr_end = int(N * 0.7)
    va_end = int(N * 0.85)
    tr_idx, va_idx, te_idx = idx[:tr_end], idx[tr_end:va_end], idx[va_end:]

    X_tr, y_tr = X[tr_idx], y[tr_idx]
    X_va, y_va = X[va_idx], y[va_idx]
    X_te, y_te = X[te_idx], y[te_idx]

    # RealMLP 전처리 파라미터는 "훈련셋의 cont 컬럼"으로 fit
    cont_idx = [i for i in range(X.shape[1]) if i not in cat_features]
    X_tr_cont = X_tr[:, cont_idx].astype("float32")
    cont_median, cont_scale = fit_robust_scale_params(X_tr_cont)

    model_params = {
        "input_dim": X.shape[1],
        "cat_features": cat_features,
        "cat_dims": cat_dims,
        "cont_median": cont_median.tolist(),
        "cont_scale": cont_scale.tolist(),
        "hidden_width": 256,
        "n_hidden_layers": 3,
        "activation": "selu",           # "mish"도 가능
        "encode_binary_as_pm1": True,
        "lr": 1e-3,
        "weight_decay": 0.0,            # RealMLP(TD-S)는 보통 weight decay 없음
        "loss_fn": "logloss",
    }

    clf = DeepLearningBinaryClassifier(model_type="realmlp", model_params=model_params)

    clf.fit(
        X_tr,
        y_tr,
        sample_weight=np.ones_like(y_tr, dtype="float32"),
        eval_set=[(X_va, y_va)],
        eval_metric=["logloss"],
        max_epochs=10,
        patience=3,
        batch_size=256,
        verbose=True,
    )

    probs_te = clf.predict_proba(X_te)[:, 1]
    preds_te = (probs_te >= 0.5).astype("int64")

    acc = (preds_te == y_te).mean()
    auc = roc_auc_score(y_te, probs_te)
    ll = log_loss(y_te, probs_te)

    print("\n===== RealMLP Test Metrics =====")
    print(f"Accuracy : {acc:.4f}")
    print(f"ROC-AUC  : {auc:.4f}")
    print(f"Logloss  : {ll:.4f}")
    print("Sample probs (first 10):", np.round(probs_te[:10], 4))


# -----------------------------------------------------------
# 메인
# -----------------------------------------------------------
if __name__ == "__main__":
    demo_train_realmlp()
    # demo_train_tabtransformer()  # 필요하면 호출



===== RealMLP Demo (KOR Feature Schema) =====
X shape: (4000, 11)  | y shape: (4000,)
Categorical feature indices: [1, 4]
Categorical dims (cardinality): [3, 4]
Epoch 1/10 - [train] loss: 0.386744
  - [eval] logloss: 0.2849
    -- (early_stopping) current_metric: 0.284884, best_metric: inf
Epoch 2/10 - [train] loss: 0.312797
  - [eval] logloss: 0.2910
    -- (early_stopping) current_metric: 0.291032, best_metric: 0.284884
Epoch 3/10 - [train] loss: 0.302301
  - [eval] logloss: 0.3021
    -- (early_stopping) current_metric: 0.302144, best_metric: 0.284884
Epoch 4/10 - [train] loss: 0.293646
  - [eval] logloss: 0.2908
    -- (early_stopping) current_metric: 0.290840, best_metric: 0.284884
Early stopping at epoch 4

===== RealMLP Test Metrics =====
Accuracy : 0.8750
ROC-AUC  : 0.8933
Logloss  : 0.2972
Sample probs (first 10): [0.1228 0.9659 0.3452 0.3108 0.9637 0.9994 0.9257 0.0143 0.9996 0.9945]


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


class RealMLPModel(nn.Module):
    """
    RealMLP-TD-S 스타일 요약:
    - categorical: one-hot (binary는 -1/+1 단일 feature 옵션)
    - continuous: robust scale + smooth clip (훈련셋으로 fit)
    - diagonal scaling + MLP
    """

    class DiagonalScaling(nn.Module):
        def __init__(self, dim: int, init: float = 1.0):
            super().__init__()
            self.scale = nn.Parameter(torch.full((dim,), float(init)))

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            return x * self.scale

    def __init__(
        self,
        input_dim: int,
        cat_features: list[int] | None = None,
        cat_dims: list[int] | None = None,
        hidden_width: int = 256,
        n_hidden_layers: int = 3,
        activation: str = "selu",        # "selu" or "mish"
        encode_binary_as_pm1: bool = True,
    ):
        super().__init__()
        self.input_dim = input_dim
        self.cat_features = cat_features or []
        self.cat_dims = cat_dims or []
        self.encode_binary_as_pm1 = encode_binary_as_pm1

        if len(self.cat_features) != len(self.cat_dims):
            raise ValueError("cat_features and cat_dims length mismatch")

        # index buffers
        self.register_buffer(
            "cat_idx_tensor",
            torch.tensor(self.cat_features, dtype=torch.long)
            if self.cat_features else torch.empty(0, dtype=torch.long),
        )
        cont_idx = [i for i in range(self.input_dim) if i not in self.cat_features]
        self.register_buffer(
            "cont_idx_tensor",
            torch.tensor(cont_idx, dtype=torch.long)
            if cont_idx else torch.empty(0, dtype=torch.long),
        )
        self.n_cont = len(cont_idx)

        # cont 전처리 파라미터 (훈련셋으로 fit 후 세팅)
        self.cont_median = None
        self.cont_scale = None

        # one-hot dim
        onehot_dim = 0
        for c in self.cat_dims:
            if self.encode_binary_as_pm1 and c == 2:
                onehot_dim += 1
            else:
                onehot_dim += int(c)
        self.onehot_dim = onehot_dim
        self.d_in = self.onehot_dim + self.n_cont

        # activation
        act = activation.lower().strip()
        if act == "selu":
            self.act_name = "selu"
            self.act = nn.SELU()
        elif act == "mish":
            self.act_name = "mish"
            self.act = nn.Mish()
        else:
            raise ValueError("activation must be 'selu' or 'mish'")

        # diagonal scaling (nested class 사용)
        self.diag = RealMLPModel.DiagonalScaling(self.d_in, init=1.0)

        # MLP
        self.mlp = self._build_mlp(hidden_width, n_hidden_layers)
        self._init_weights()

    # ----- (클래스 내부 유틸) numpy smooth clip -----
    @staticmethod
    def smooth_clip_np(x: np.ndarray) -> np.ndarray:
        return x / np.power(1.0 + np.abs(x) ** 3, 2.0 / 3.0)

    # ----- (클래스 내부 유틸) robust scale fit -----
    @staticmethod
    def fit_robust_scale_params(X_cont: np.ndarray, eps: float = 1e-12):
        q0  = np.quantile(X_cont, 0.00, axis=0)
        q25 = np.quantile(X_cont, 0.25, axis=0)
        q50 = np.quantile(X_cont, 0.50, axis=0)
        q75 = np.quantile(X_cont, 0.75, axis=0)

        iqr = q75 - q25
        scale = np.zeros_like(iqr, dtype=np.float32)

        mask1 = np.abs(iqr) > eps
        scale[mask1] = 1.0 / iqr[mask1]

        mask2 = (~mask1) & (np.abs(q50 - q0) > eps)
        scale[mask2] = 2.0 / (q50[mask2] - q0[mask2])

        median = q50.astype(np.float32)
        scale = scale.astype(np.float32)
        return median, scale

    # ----- (권장) 훈련 데이터로 cont 전처리 파라미터 fit 후 버퍼 저장 -----
    def fit_cont_preprocess_from_numpy(self, X_train: np.ndarray, eps: float = 1e-12) -> None:
        if self.n_cont == 0:
            return
        cont_idx = self.cont_idx_tensor.cpu().numpy().tolist()
        X_cont = X_train[:, cont_idx].astype(np.float32)

        median, scale = self.fit_robust_scale_params(X_cont, eps=eps)

        median_t = torch.tensor(median, dtype=torch.float32)
        scale_t = torch.tensor(scale, dtype=torch.float32)

        if hasattr(self, "_cont_median_buf"):
            self._cont_median_buf.data = median_t
            self._cont_scale_buf.data = scale_t
        else:
            self.register_buffer("_cont_median_buf", median_t)
            self.register_buffer("_cont_scale_buf", scale_t)

        self.cont_median = self._cont_median_buf
        self.cont_scale = self._cont_scale_buf

    @staticmethod
    def _smooth_clip_torch(x: torch.Tensor) -> torch.Tensor:
        return x / torch.pow(1.0 + torch.abs(x) ** 3, 2.0 / 3.0)

    def _preprocess_cont(self, x_cont: torch.Tensor) -> torch.Tensor:
        if x_cont.numel() == 0:
            return x_cont
        if self.cont_median is None or self.cont_scale is None:
            # fit 전이라면 no-op처럼 동작 (원하면 여기서 raise로 바꿔도 됨)
            return self._smooth_clip_torch(x_cont)
        xc = (x_cont - self.cont_median) * self.cont_scale
        return self._smooth_clip_torch(xc)

    def _encode_categorical_onehot(self, x_cat: torch.Tensor) -> torch.Tensor:
        if x_cat.numel() == 0:
            return torch.zeros((x_cat.shape[0], 0), device=x_cat.device, dtype=torch.float32)

        outs = []
        for j, card in enumerate(self.cat_dims):
            col = x_cat[:, j]
            if self.encode_binary_as_pm1 and card == 2:
                pm1 = col.to(torch.float32) * 2.0 - 1.0
                outs.append(pm1.unsqueeze(1))
            else:
                oh = F.one_hot(col.clamp(min=0, max=card - 1), num_classes=card).to(torch.float32)
                outs.append(oh)
        return torch.cat(outs, dim=1)

    def _build_mlp(self, hidden_width: int, n_hidden_layers: int) -> nn.Module:
        layers: list[nn.Module] = []
        in_dim = self.d_in
        for _ in range(n_hidden_layers):
            layers.append(nn.Linear(in_dim, hidden_width))
            layers.append(self.act)
            in_dim = hidden_width
        layers.append(nn.Linear(in_dim, 1))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                if self.act_name == "selu":
                    nn.init.normal_(m.weight, mean=0.0, std=(1.0 / m.in_features) ** 0.5)
                else:
                    nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B = x.shape[0]
        n_cat = len(self.cat_features)

        x_cat = x[:, self.cat_idx_tensor].long() if n_cat > 0 else torch.zeros((B, 0), device=x.device, dtype=torch.long)

        if self.n_cont > 0:
            x_cont = x[:, self.cont_idx_tensor].to(torch.float32)
            x_cont = self._preprocess_cont(x_cont)
        else:
            x_cont = torch.zeros((B, 0), device=x.device, dtype=torch.float32)

        x_oh = self._encode_categorical_onehot(x_cat)
        feats = torch.cat([x_oh, x_cont], dim=1)
        feats = self.diag(feats)
        return self.mlp(feats)


In [1]:
import math
import torch
import torch.nn as nn
from abc import ABC, abstractmethod


# -----------------------------------------------------------
# RealMLP Model (single class)
# -----------------------------------------------------------
class RealMLPModel(nn.Module):
    """
    RealMLP-style MLP for tabular data (network part).
    - Supports:
      * categorical embeddings (optional)
      * numerical embeddings: PBLD (optional)
      * scaling layer (learnable per-feature diagonal)
      * parametric activation (optional)
      * dropout (optional)

    Notes
    -----
    - The original RealMLP paper uses strong preprocessing (robust scaling + smooth clipping)
      and one-hot encoding for small-cardinality categoricals (and for TD-S: one-hot for all categoricals).
      That preprocessing is NOT included here because it needs dataset statistics (quantiles).
    """

    # ---------- small helpers ----------
    class _Mish(nn.Module):
        def forward(self, x: torch.Tensor) -> torch.Tensor:
            return x * torch.tanh(nn.functional.softplus(x))

    class _ParametricAct(nn.Module):
        """
        sigma_alpha_i(x_i) = (1-alpha_i) * x_i + alpha_i * sigma(x_i)
        (alpha is learnable per neuron; init=1 => standard activation)
        """
        def __init__(self, n_units: int, base_act: nn.Module, alpha_init: float = 1.0):
            super().__init__()
            self.base_act = base_act
            self.alpha = nn.Parameter(torch.full((n_units,), float(alpha_init)))

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            # x: (B, n_units)
            a = self.alpha.view(1, -1)
            return (1.0 - a) * x + a * self.base_act(x)

    class _PBLDNumEmbedding(nn.Module):
        """
        PBLD embedding (paper): concat(x, W2 cos(2π w1 x + b1) + b2) in R^{1 + d_periodic}
        We implement d_periodic=3 by default => output dim 4 per numerical feature.
        """
        def __init__(
            self,
            n_features: int,
            d_periodic: int = 3,
            periodic_init_std: float = 0.1,
        ):
            super().__init__()
            self.n_features = n_features
            self.d_periodic = d_periodic

            # Per-feature parameters:
            # w1, b1: (n_features, d_periodic)
            # W2:     (n_features, d_periodic, d_periodic)
            # b2:     (n_features, d_periodic)
            self.w1 = nn.Parameter(torch.randn(n_features, d_periodic) * periodic_init_std)
            self.b1 = nn.Parameter(torch.zeros(n_features, d_periodic))
            self.W2 = nn.Parameter(torch.eye(d_periodic).unsqueeze(0).repeat(n_features, 1, 1))
            self.b2 = nn.Parameter(torch.zeros(n_features, d_periodic))

        def forward(self, x_cont: torch.Tensor) -> torch.Tensor:
            """
            x_cont: (B, n_features) float
            return: (B, n_features * (1 + d_periodic))
            """
            B, F = x_cont.shape
            assert F == self.n_features

            # (B, F, 1)
            x_ = x_cont.unsqueeze(-1)

            # (B, F, d)
            phase = 2.0 * math.pi * (x_ * self.w1.unsqueeze(0)) + self.b1.unsqueeze(0)
            z = torch.cos(phase)

            # (B, F, d) = (B,F,d) @ (F,d,d)^T per-feature
            # einsum: b f d , f d k -> b f k
            z2 = torch.einsum("bfd,fdk->bfk", z, self.W2) + self.b2.unsqueeze(0)

            # concat original x with periodic projection
            out = torch.cat([x_, z2], dim=-1)  # (B, F, 1+d)
            return out.reshape(B, -1)

    # ---------- main ----------
    def __init__(
        self,
        input_dim: int,
        cat_features: list[int] | None = None,
        cat_dims: list[int] | None = None,
        cat_emb_dim: int = 8,
        # numerical embedding
        num_embedding: str = "pbld",   # "pbld" or "none"
        pbld_d_periodic: int = 3,      # => per-feature dim = 1 + d_periodic (default 4)
        pbld_init_std: float = 0.1,
        # MLP
        hidden_dims: tuple[int, ...] = (256, 256, 256),
        activation: str = "selu",      # "selu" or "mish" (paper: SELU for clf, Mish for reg)
        parametric_activation: bool = False,
        dropout: float = 0.1,
        # scaling layer
        use_scaling: bool = True,
    ):
        super().__init__()

        self.input_dim = input_dim
        self.cat_features = cat_features or []
        self.cat_dims = cat_dims or []
        self.cat_emb_dim = cat_emb_dim

        if len(self.cat_features) != len(self.cat_dims):
            raise ValueError(
                f"Mismatch between cat_features({len(self.cat_features)}) and cat_dims({len(self.cat_dims)})"
            )

        self.num_embedding = num_embedding
        self.pbld_d_periodic = pbld_d_periodic
        self.pbld_init_std = pbld_init_std

        self.hidden_dims = hidden_dims
        self.activation = activation.lower()
        self.parametric_activation = parametric_activation
        self.dropout = float(dropout)
        self.use_scaling = use_scaling

        # indices
        cont_idx = [i for i in range(self.input_dim) if i not in self.cat_features]
        self.register_buffer("cat_idx_tensor", torch.tensor(self.cat_features, dtype=torch.long) if self.cat_features else torch.empty(0, dtype=torch.long))
        self.register_buffer("cont_idx_tensor", torch.tensor(cont_idx, dtype=torch.long) if cont_idx else torch.empty(0, dtype=torch.long))

        self.n_cat = len(self.cat_features)
        self.n_cont = len(cont_idx)

        # categorical embeddings
        if self.n_cat > 0:
            self.cat_embeddings = nn.ModuleList(
                [nn.Embedding(num_embeddings=c, embedding_dim=self.cat_emb_dim) for c in self.cat_dims]
            )
            for emb in self.cat_embeddings:
                nn.init.normal_(emb.weight, std=0.02)
        else:
            self.cat_embeddings = nn.ModuleList()

        # numerical embeddings
        if self.n_cont > 0 and self.num_embedding == "pbld":
            self.num_emb = RealMLPModel._PBLDNumEmbedding(
                n_features=self.n_cont,
                d_periodic=self.pbld_d_periodic,
                periodic_init_std=self.pbld_init_std,
            )
            self.per_cont_dim = 1 + self.pbld_d_periodic
        else:
            self.num_emb = None
            self.per_cont_dim = 1

        # combined input dimension to MLP
        mlp_in_dim = self.n_cat * self.cat_emb_dim + self.n_cont * self.per_cont_dim

        # scaling layer (diagonal weights)
        if self.use_scaling:
            self.scale = nn.Parameter(torch.ones(mlp_in_dim))
        else:
            self.scale = None

        # activation module
        if self.activation == "selu":
            base_act = nn.SELU()
            init_mode = "lecun"
        elif self.activation == "mish":
            base_act = RealMLPModel._Mish()
            init_mode = "kaiming"
        else:
            raise ValueError("activation must be one of: 'selu', 'mish'")

        # build MLP: Linear -> Act -> Dropout (repeat) -> Linear(out)
        layers: list[nn.Module] = []
        dims = [mlp_in_dim] + list(self.hidden_dims) + [1]

        for i in range(len(dims) - 2):
            lin = nn.Linear(dims[i], dims[i + 1])
            self._init_linear(lin, init_mode=init_mode)
            layers.append(lin)

            if self.parametric_activation:
                layers.append(RealMLPModel._ParametricAct(dims[i + 1], base_act, alpha_init=1.0))
            else:
                layers.append(base_act)

            if self.dropout > 0:
                layers.append(nn.Dropout(self.dropout))

        out_lin = nn.Linear(dims[-2], dims[-1])
        self._init_linear(out_lin, init_mode=init_mode, is_last=True)
        layers.append(out_lin)

        self.mlp = nn.Sequential(*layers)

    @staticmethod
    def _init_linear(lin: nn.Linear, init_mode: str = "lecun", is_last: bool = False) -> None:
        # Paper uses data-dependent init; here we use robust defaults.
        fan_in = lin.weight.size(1)

        if init_mode == "lecun":
            # LeCun normal-ish (good for SELU)
            std = 1.0 / math.sqrt(fan_in)
            nn.init.normal_(lin.weight, mean=0.0, std=std)
        else:
            # Kaiming for general activations
            nn.init.kaiming_normal_(lin.weight, nonlinearity="relu")

        nn.init.zeros_(lin.bias)

        # Optionally make last layer small (sometimes stabilizes early training)
        if is_last:
            nn.init.zeros_(lin.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, input_dim) float tensor
        returns logits: (B, 1)
        """
        B = x.size(0)

        # categorical
        if self.n_cat > 0:
            x_cat = x[:, self.cat_idx_tensor].long()  # assumes already integer-coded
            cat_embs = []
            for j, emb in enumerate(self.cat_embeddings):
                cat_embs.append(emb(x_cat[:, j]))
            z_cat = torch.cat(cat_embs, dim=1)  # (B, n_cat*emb_dim)
        else:
            z_cat = x.new_zeros((B, 0))

        # continuous
        if self.n_cont > 0:
            x_cont = x[:, self.cont_idx_tensor].float()
            if self.num_emb is not None:
                z_cont = self.num_emb(x_cont)  # (B, n_cont*(1+d))
            else:
                z_cont = x_cont  # (B, n_cont)
        else:
            z_cont = x.new_zeros((B, 0))

        z = torch.cat([z_cat, z_cont], dim=1)

        # scaling layer (diagonal)
        if self.scale is not None:
            z = z * self.scale.view(1, -1)

        logits = self.mlp(z)
        return logits

In [3]:
# model_registry에 추가
model_registry = {
    "mlp": MLPModel,
    "tabtransformer": TabTransformerModel,
    "realmlp": RealMLPModel,   # <= 추가
}

model_params = {
    "input_dim": input_dim,
    "cat_features": cat_features,
    "cat_dims": cat_dims,

    "cat_emb_dim": 8,
    "num_embedding": "pbld",          # "none"도 가능
    "pbld_d_periodic": 3,             # => 4-dim/feature
    "pbld_init_std": 0.1,

    "hidden_dims": (256, 256, 256),
    "activation": "selu",
    "parametric_activation": False,   # True로도 가능(논문: regression에 특히 유리) :contentReference[oaicite:3]{index=3}
    "dropout": 0.1,
    "use_scaling": True,

    "lr": 1e-3,
    "loss_fn": "logloss",
}


In [4]:
# -*- coding: utf-8 -*-
# ===========================================================
# Demo: RealMLP (single-class) + existing DeepLearningBinaryClassifier
# ===========================================================

from abc import ABC, abstractmethod
import copy
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score
from torch.utils.data import DataLoader, TensorDataset


# -----------------------------------------------------------
# Base Neural Network Model
# -----------------------------------------------------------
class BaseNNModel(nn.Module, ABC):
    @abstractmethod
    def __init__(self, **kwargs):
        super(BaseNNModel, self).__init__()

    @abstractmethod
    def build_network(self) -> nn.Module:
        pass

    @abstractmethod
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        pass


# -----------------------------------------------------------
# RealMLP Model (SINGLE class, no nested classes)
# -----------------------------------------------------------
class RealMLPModel(BaseNNModel):
    """
    RealMLP-style MLP (network part) in a single class.

    Supports:
      - categorical embeddings (optional)
      - numerical embedding: PBLD (optional)
      - scaling layer (learnable per-feature diagonal)
      - parametric activation (optional)
      - dropout (optional)

    Note:
      - RealMLP paper's strong preprocessing (robust scaling + smooth clipping) is NOT included.
        It needs dataset statistics (quantiles), better handled in preprocessing pipeline or estimator-level fit().
    """

    def __init__(
        self,
        input_dim: int,
        cat_features: list[int] | None = None,
        cat_dims: list[int] | None = None,
        cat_emb_dim: int = 8,
        # numerical embedding
        num_embedding: str = "pbld",   # "pbld" or "none"
        pbld_d_periodic: int = 3,      # per numeric feature dim = 1 + d_periodic
        pbld_init_std: float = 0.1,
        # MLP
        hidden_dims: tuple[int, ...] = (256, 256, 256),
        activation: str = "selu",      # "selu" or "mish"
        parametric_activation: bool = False,
        dropout: float = 0.1,
        # scaling layer
        use_scaling: bool = True,
    ):
        super().__init__()

        self.input_dim = input_dim
        self.cat_features = cat_features or []
        self.cat_dims = cat_dims or []
        self.cat_emb_dim = int(cat_emb_dim)

        if len(self.cat_features) != len(self.cat_dims):
            raise ValueError(
                f"Mismatch between cat_features({len(self.cat_features)}) and cat_dims({len(self.cat_dims)})"
            )

        self.num_embedding = num_embedding
        self.pbld_d_periodic = int(pbld_d_periodic)
        self.pbld_init_std = float(pbld_init_std)

        self.hidden_dims = tuple(int(h) for h in hidden_dims)
        self.activation_name = activation.lower()
        self.parametric_activation = bool(parametric_activation)
        self.dropout = float(dropout)
        self.use_scaling = bool(use_scaling)

        # indices
        cont_idx = [i for i in range(self.input_dim) if i not in self.cat_features]
        self.n_cat = len(self.cat_features)
        self.n_cont = len(cont_idx)

        self.register_buffer(
            "cat_idx_tensor",
            torch.tensor(self.cat_features, dtype=torch.long) if self.n_cat > 0 else torch.empty(0, dtype=torch.long),
        )
        self.register_buffer(
            "cont_idx_tensor",
            torch.tensor(cont_idx, dtype=torch.long) if self.n_cont > 0 else torch.empty(0, dtype=torch.long),
        )

        # placeholders
        self.cat_embeddings = None
        self.pbld_w1 = None
        self.pbld_b1 = None
        self.pbld_W2 = None
        self.pbld_b2 = None
        self.scale = None
        self.alphas = None
        self.linears = None

        self.build_network()

    def build_network(self) -> None:
        # categorical embeddings
        if self.n_cat > 0:
            self.cat_embeddings = nn.ModuleList(
                [nn.Embedding(num_embeddings=c, embedding_dim=self.cat_emb_dim) for c in self.cat_dims]
            )
            for emb in self.cat_embeddings:
                nn.init.normal_(emb.weight, std=0.02)
        else:
            self.cat_embeddings = nn.ModuleList()

        # numerical embedding params (PBLD)
        if self.n_cont > 0 and self.num_embedding == "pbld":
            d = self.pbld_d_periodic
            self.pbld_w1 = nn.Parameter(torch.randn(self.n_cont, d) * self.pbld_init_std)
            self.pbld_b1 = nn.Parameter(torch.zeros(self.n_cont, d))
            self.pbld_W2 = nn.Parameter(torch.eye(d).unsqueeze(0).repeat(self.n_cont, 1, 1))
            self.pbld_b2 = nn.Parameter(torch.zeros(self.n_cont, d))
            self.per_cont_dim = 1 + d
        else:
            self.per_cont_dim = 1

        # combined input dim
        mlp_in_dim = self.n_cat * self.cat_emb_dim + self.n_cont * self.per_cont_dim

        # scaling layer (diagonal)
        self.scale = nn.Parameter(torch.ones(mlp_in_dim)) if self.use_scaling else None

        # activation
        if self.activation_name not in ("selu", "mish"):
            raise ValueError("activation must be one of: 'selu', 'mish'")

        # parametric activation alphas
        if self.parametric_activation:
            self.alphas = nn.ParameterList([nn.Parameter(torch.ones(h)) for h in self.hidden_dims])
        else:
            self.alphas = nn.ParameterList()

        # MLP
        dims = [mlp_in_dim] + list(self.hidden_dims) + [1]
        self.linears = nn.ModuleList([nn.Linear(dims[i], dims[i + 1]) for i in range(len(dims) - 1)])

        init_mode = "lecun" if self.activation_name == "selu" else "kaiming"
        for i, lin in enumerate(self.linears):
            is_last = (i == len(self.linears) - 1)
            self._init_linear(lin, init_mode=init_mode, is_last=is_last)

    def _init_linear(self, lin: nn.Linear, init_mode: str, is_last: bool) -> None:
        fan_in = lin.weight.size(1)
        if init_mode == "lecun":
            std = 1.0 / math.sqrt(fan_in)
            nn.init.normal_(lin.weight, mean=0.0, std=std)
        else:
            nn.init.kaiming_normal_(lin.weight, nonlinearity="relu")
        nn.init.zeros_(lin.bias)
        if is_last:
            nn.init.zeros_(lin.weight)

    def _encode_categoricals(self, x: torch.Tensor) -> torch.Tensor:
        if self.n_cat == 0:
            return x.new_zeros((x.size(0), 0))
        x_cat = x[:, self.cat_idx_tensor].long()
        embs = [emb(x_cat[:, j]) for j, emb in enumerate(self.cat_embeddings)]
        return torch.cat(embs, dim=1)

    def _encode_continuous(self, x: torch.Tensor) -> torch.Tensor:
        if self.n_cont == 0:
            return x.new_zeros((x.size(0), 0))
        x_cont = x[:, self.cont_idx_tensor].float()

        if self.num_embedding != "pbld":
            return x_cont

        # PBLD
        x_ = x_cont.unsqueeze(-1)  # (B,F,1)
        phase = 2.0 * math.pi * (x_ * self.pbld_w1.unsqueeze(0)) + self.pbld_b1.unsqueeze(0)  # (B,F,d)
        z = torch.cos(phase)  # (B,F,d)
        z2 = torch.einsum("bfd,fdk->bfk", z, self.pbld_W2) + self.pbld_b2.unsqueeze(0)  # (B,F,d)
        out = torch.cat([x_, z2], dim=-1)  # (B,F,1+d)
        return out.reshape(x.size(0), -1)

    def _base_activation(self, x: torch.Tensor) -> torch.Tensor:
        if self.activation_name == "selu":
            return F.selu(x)
        # mish
        return x * torch.tanh(F.softplus(x))

    def _apply_activation(self, x: torch.Tensor, layer_idx: int) -> torch.Tensor:
        if not self.parametric_activation:
            return self._base_activation(x)
        a = self.alphas[layer_idx].view(1, -1)
        return (1.0 - a) * x + a * self._base_activation(x)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z_cat = self._encode_categoricals(x)
        z_cont = self._encode_continuous(x)
        z = torch.cat([z_cat, z_cont], dim=1)

        if self.scale is not None:
            z = z * self.scale.view(1, -1)

        h = z
        n_hidden = len(self.hidden_dims)
        for i in range(n_hidden):
            h = self.linears[i](h)
            h = self._apply_activation(h, layer_idx=i)
            if self.dropout > 0:
                h = F.dropout(h, p=self.dropout, training=self.training)

        logits = self.linears[-1](h)
        return logits


# -----------------------------------------------------------
# Deep Learning Binary Classifier
# -----------------------------------------------------------
class DeepLearningBinaryClassifier(BaseEstimator, ClassifierMixin):
    def __init__(
        self,
        model_type: str = "mlp",
        model_params: dict | None = None,
    ):
        self.model_type = model_type
        self.model_params = model_params or {}
        self.model = None

    @property
    def device(self) -> torch.device:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def _build_model(self) -> BaseNNModel:
        model_registry = {
            # 기존에 있던 모델들은 생략해도 되지만,
            # demo에서는 realmlp만 쓰므로 realmlp만 둬도 동작합니다.
            "realmlp": RealMLPModel,
        }

        if self.model_type not in model_registry:
            raise ValueError(f"Unknown model type: {self.model_type}")

        valid_params = {k: v for k, v in self.model_params.items() if k not in ["loss_fn", "lr"]}
        model = model_registry[self.model_type](**valid_params)
        return model

    def _get_loss_fn(self) -> nn.Module:
        loss_name = self.model_params.get("loss_fn", "logloss")
        if loss_name == "logloss":
            return nn.BCEWithLogitsLoss(reduction="none")
        else:
            raise ValueError(f"Unknown loss function: {loss_name}")

    def fit(
        self,
        X: np.ndarray,
        y: np.ndarray,
        sample_weight: np.ndarray | None = None,
        eval_set: list[tuple[np.ndarray, np.ndarray]] | None = None,
        eval_metric: list[str] | None = None,
        max_epochs: int = 10,
        patience: int | None = None,
        batch_size: int = 128,
        verbose: bool = True,
    ) -> "DeepLearningBinaryClassifier":

        lr = self.model_params.get("lr", 0.001)
        eval_metric = eval_metric or ["logloss"]

        X_tensor = torch.tensor(X, dtype=torch.float32).to(self.device)
        y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1).to(self.device)

        if eval_set is not None:
            x_eval_tensor = torch.tensor(eval_set[0][0], dtype=torch.float32).to(self.device)
            y_eval_true = eval_set[0][1]
        else:
            x_eval_tensor = None
            y_eval_true = None

        if sample_weight is not None:
            sample_weight_tensor = torch.tensor(sample_weight, dtype=torch.float32).view(-1, 1).to(self.device)
        else:
            sample_weight_tensor = torch.ones_like(y_tensor, dtype=torch.float32)

        train_dataset = TensorDataset(X_tensor, y_tensor, sample_weight_tensor)
        train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

        if self.model is None:
            self.model = self._build_model().to(self.device)

        loss_fn = self._get_loss_fn()
        optimizer = optim.Adam(self.model.parameters(), lr=lr, weight_decay=1e-4)

        patience_counter = 0
        best_metric = float("inf")
        best_model_weights = None

        for epoch in range(max_epochs):
            self.model.train()
            epoch_loss = 0.0
            n_batches = 0

            for x_batch, y_batch, weight_batch in train_dataloader:
                optimizer.zero_grad()
                y_pred_logits = self.model(x_batch)

                loss = loss_fn(y_pred_logits, y_batch)
                weighted_loss = (loss * weight_batch).sum() / weight_batch.sum()

                weighted_loss.backward()
                optimizer.step()

                epoch_loss += weighted_loss.item()
                n_batches += 1

            if verbose:
                print(f"Epoch {epoch + 1}/{max_epochs} - [train] loss: {epoch_loss / max(1, n_batches):.6f}")

            # evaluation
            if eval_set is not None:
                self.model.eval()
                with torch.no_grad():
                    y_eval_logits = self.model(x_eval_tensor)
                    y_eval_pred = torch.sigmoid(y_eval_logits).cpu().numpy().ravel()

                eval_metrics = {}
                for metric in eval_metric:
                    if metric == "logloss":
                        eval_metrics["logloss"] = log_loss(y_eval_true, y_eval_pred)
                    elif metric == "average_precision":
                        eval_metrics["average_precision"] = -average_precision_score(y_eval_true, y_eval_pred)
                    elif metric == "auc":
                        eval_metrics["auc"] = -roc_auc_score(y_eval_true, y_eval_pred)
                    else:
                        raise ValueError(f"Unknown metric: {metric}")

                if verbose:
                    metrics_str = ", ".join([f"{k}: {v:.4f}" for k, v in eval_metrics.items()])
                    print(f"  - [eval] {metrics_str}")

                main_metric_name = eval_metric[0]
                current_metric = eval_metrics.get(main_metric_name, eval_metrics["logloss"])

                if verbose:
                    print(f"    -- (early_stopping) current_metric: {current_metric:.6f}, best_metric: {best_metric:.6f}")

                if current_metric < best_metric:
                    best_metric = current_metric
                    patience_counter = 0
                    best_model_weights = copy.deepcopy(self.model.state_dict())
                else:
                    patience_counter += 1
                    if patience is not None and patience_counter >= patience:
                        if verbose:
                            print(f"Early stopping at epoch {epoch + 1}")
                        break

        if best_model_weights is not None:
            self.model.load_state_dict(best_model_weights)

        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        self.model = self.model.to(self.device)
        X_tensor = torch.tensor(X, dtype=torch.float32).to(self.device)

        with torch.no_grad():
            self.model.eval()
            logits = self.model(X_tensor)
            probs1 = torch.sigmoid(logits).cpu().numpy()

        if probs1.shape[1] == 1:
            probs1 = probs1.reshape(-1, 1)

        probs0 = 1.0 - probs1
        probs = np.hstack((probs0, probs1))
        return probs.astype("float")

    def predict(self, X: np.ndarray) -> np.ndarray:
        probs = self.predict_proba(X)
        return probs.argmax(axis=1)


# -----------------------------------------------------------
# Demo data generator (same as yours)
# -----------------------------------------------------------
def make_kor_feature_demo_data(n_samples: int = 5000, seed: int = 42):
    rng = np.random.RandomState(seed)
    n = n_samples

    age = np.clip(rng.normal(loc=40, scale=12, size=n), 18, 80).astype("float32")
    gender = rng.randint(0, 3, size=n).astype("int64")
    app_join_cnt = rng.poisson(lam=2.0, size=n).astype("float32")
    last_login_days = rng.exponential(scale=10.0, size=n).astype("float32")
    os_cat = rng.randint(0, 4, size=n).astype("int64")
    app_join_days = rng.exponential(scale=200.0, size=n).astype("float32")

    biz_owner_score = (rng.binomial(1, 0.3, size=n) * rng.uniform(0.5, 1.0, size=n)).astype("float32")
    home_owner_score = (rng.binomial(1, 0.4, size=n) * rng.uniform(0.5, 1.0, size=n)).astype("float32")
    car_owner_score = (rng.binomial(1, 0.5, size=n) * rng.uniform(0.5, 1.0, size=n)).astype("float32")
    married_score   = (rng.binomial(1, 0.5, size=n) * rng.uniform(0.5, 1.0, size=n)).astype("float32")
    child_score     = (rng.binomial(1, 0.4, size=n) * rng.uniform(0.5, 1.0, size=n)).astype("float32")

    w_gender = rng.uniform(-0.3, 0.3, size=3)
    w_os     = rng.uniform(-0.2, 0.2, size=4)

    score = np.zeros(n, dtype="float32")
    score += 0.05 * (age - 40) / 10.0
    score += 0.25 * app_join_cnt
    score -= 0.03 * last_login_days
    score -= 0.002 * app_join_days
    score += 0.8 * biz_owner_score
    score += 0.6 * home_owner_score
    score += 0.7 * car_owner_score
    score += 0.9 * married_score
    score += 1.0 * child_score
    score += w_gender[gender]
    score += w_os[os_cat]

    noise = rng.normal(scale=0.5, size=n).astype("float32")
    bias = -0.2
    logit = score + bias + noise
    prob = 1.0 / (1.0 + np.exp(-logit))
    y = (prob > 0.5).astype("int64")

    X = np.column_stack(
        [
            age,                      # 0
            gender.astype("float32"), # 1
            app_join_cnt,             # 2
            last_login_days,          # 3
            os_cat.astype("float32"), # 4
            app_join_days,            # 5
            biz_owner_score,          # 6
            home_owner_score,         # 7
            car_owner_score,          # 8
            married_score,            # 9
            child_score,              # 10
        ]
    ).astype("float32")

    cat_feature_indices = [1, 4]
    cat_dims = [3, 4]
    return X, y, cat_feature_indices, cat_dims


# -----------------------------------------------------------
# RealMLP demo
# -----------------------------------------------------------
def demo_train_realmlp():
    print("\n===== RealMLP Demo (KOR Feature Schema) =====")

    X, y, cat_features, cat_dims = make_kor_feature_demo_data(n_samples=2000, seed=777)

    print("X shape:", X.shape, " | y shape:", y.shape)
    print("Categorical feature indices:", cat_features)
    print("Categorical dims (cardinality):", cat_dims)

    # train / val / test split
    N = X.shape[0]
    idx = np.arange(N)
    np.random.shuffle(idx)

    tr_end = int(N * 0.7)
    va_end = int(N * 0.85)
    tr_idx, va_idx, te_idx = idx[:tr_end], idx[tr_end:va_end], idx[va_end:]

    X_tr, y_tr = X[tr_idx], y[tr_idx]
    X_va, y_va = X[va_idx], y[va_idx]
    X_te, y_te = X[te_idx], y[te_idx]

    sample_weight = np.ones_like(y_tr, dtype="float32")
    input_dim = X.shape[1]

    model_params = {
        "input_dim": input_dim,
        "cat_features": cat_features,
        "cat_dims": cat_dims,

        "cat_emb_dim": 8,
        "num_embedding": "pbld",        # "none"로 바꿔도 됨
        "pbld_d_periodic": 3,           # per num feature dim = 4
        "pbld_init_std": 0.1,

        "hidden_dims": (256, 256, 256),
        "activation": "selu",
        "parametric_activation": False, # True로도 테스트 가능
        "dropout": 0.1,
        "use_scaling": True,

        "lr": 1e-3,
        "loss_fn": "logloss",
    }

    clf = DeepLearningBinaryClassifier(
        model_type="realmlp",
        model_params=model_params,
    )

    clf.fit(
        X_tr,
        y_tr,
        sample_weight=sample_weight,
        eval_set=[(X_va, y_va)],
        eval_metric=["logloss"],
        max_epochs=10,
        patience=3,
        batch_size=256,
        verbose=True,
    )

    probs_te = clf.predict_proba(X_te)[:, 1]
    preds_te = (probs_te >= 0.5).astype("int64")

    acc = (preds_te == y_te).mean()
    auc = roc_auc_score(y_te, probs_te)
    ll = log_loss(y_te, probs_te)

    print("\n===== RealMLP Test Metrics =====")
    print(f"Accuracy : {acc:.4f}")
    print(f"ROC-AUC  : {auc:.4f}")
    print(f"Logloss  : {ll:.4f}")
    print("Sample probs (first 10):", np.round(probs_te[:10], 4))


if __name__ == "__main__":
    demo_train_realmlp()



===== RealMLP Demo (KOR Feature Schema) =====
X shape: (2000, 11)  | y shape: (2000,)
Categorical feature indices: [1, 4]
Categorical dims (cardinality): [3, 4]
Epoch 1/10 - [train] loss: 0.656029
  - [eval] logloss: 0.5747
    -- (early_stopping) current_metric: 0.574723, best_metric: inf
Epoch 2/10 - [train] loss: 0.514563
  - [eval] logloss: 0.5630
    -- (early_stopping) current_metric: 0.562983, best_metric: 0.574723
Epoch 3/10 - [train] loss: 0.492870
  - [eval] logloss: 0.5412
    -- (early_stopping) current_metric: 0.541207, best_metric: 0.562983
Epoch 4/10 - [train] loss: 0.495647
  - [eval] logloss: 0.5148
    -- (early_stopping) current_metric: 0.514780, best_metric: 0.541207
Epoch 5/10 - [train] loss: 0.498840
  - [eval] logloss: 0.5280
    -- (early_stopping) current_metric: 0.527958, best_metric: 0.514780
Epoch 6/10 - [train] loss: 0.489018
  - [eval] logloss: 0.5185
    -- (early_stopping) current_metric: 0.518477, best_metric: 0.514780
Epoch 7/10 - [train] loss: 0.4781